---
## เคส B — 3B + LoRA เต็ม 16-bit

โมเดล: `typhoon-ai/llama3.2-typhoon2-3b-instruct` — ไม่ quantize เพราะ 3B × 2 bytes = ~6 GB พอดี VRAM T4

**ทำไมไม่ใช้ Gemma ตามแผนเดิม** (`google/gemma-4-E4B-it`): Gemma ต้องใช้ **bf16** เพราะ activation มีค่าสูงจนทะลุเพดาน fp16 (65,504) แต่ T4 เป็นสถาปัตยกรรม Turing (2018) ซึ่ง**ไม่รองรับ bf16** (มาใน Ampere ปี 2020) Unsloth เลยถอยไปใช้ fp32 = 4 bytes/param → ~16 GB → **OOM ตั้งแต่โหลดโมเดล** แก้ด้วยการลด batch/seq len ไม่ได้เพราะน้ำหนักโมเดลอย่างเดียวก็เกินแล้ว

→ ต้องเลี่ยงโมเดลฐาน Gemma ทุกตัวบน T4 รวมถึง `typhoon2.1-gemma3-4b` ด้วย

Llama 3.2 เทรนบน NVIDIA GPU มาแต่แรก ใช้ fp16 ได้ปกติ และ Typhoon ผ่านการเทรนภาษาไทยมาแล้ว เหมาะกับงานนี้กว่า Qwen2.5-3B ดิบๆ


## 0. ติดตั้ง + เตรียม environment

In [ ]:
%%capture
import os
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers triton

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/thai-paper-feed-phase-b'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints'
ADAPTER_DIR = f'{DRIVE_ROOT}/adapters'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

## 1. โหลด dataset + แบ่ง tuning split

อัปโหลด `train.jsonl` / `test.jsonl` ขึ้น Drive ที่ `DRIVE_ROOT/data/` ก่อนรันเซลล์นี้

`tune_dev` ถูกแบ่งออกมาจาก **train.jsonl เท่านั้น** ใช้แค่ตอน grid search — `test.jsonl` ทั้ง 60 ใบเก็บไว้ไม่แตะจนถึง Stage 3

In [ ]:
import json, random

DATA_DIR = f'{DRIVE_ROOT}/data'  # ต้องมี train.jsonl และ test.jsonl อยู่ในนี้

def load_jsonl(path):
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(f'{DATA_DIR}/train.jsonl')
test_rows = load_jsonl(f'{DATA_DIR}/test.jsonl')
print(f'train: {len(train_rows)} rows, test: {len(test_rows)} rows (held out for Stage 3)')

def make_tune_split(rows, dev_frac=0.2, seed=42):
    shuffled = rows.copy()
    random.Random(seed).shuffle(shuffled)
    n_dev = max(1, int(len(shuffled) * dev_frac))
    return shuffled[n_dev:], shuffled[:n_dev]  # (tune_train, tune_dev)

tune_train_rows, tune_dev_rows = make_tune_split(train_rows)
print(f'tune_train: {len(tune_train_rows)}, tune_dev: {len(tune_dev_rows)} (สำหรับ grid search เท่านั้น)')

In [ ]:
def to_chat_messages(row):
    return [
        {"role": "system", "content": row["system"]},
        {"role": "user", "content": row["user"]},
        {"role": "assistant", "content": row["assistant"]},
    ]

def build_formatting_func(tokenizer):
    def format_examples(examples):
        texts = []
        n = len(examples["system"])
        for i in range(n):
            row = {k: examples[k][i] for k in examples}
            messages = to_chat_messages(row)
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            texts.append(text)
        return {"text": texts}
    return format_examples

In [ ]:
import gc, torch
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"]

def load_base_model(model_name, max_seq_length, load_in_4bit):
    return FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=load_in_4bit,
    )

def add_lora(model, r, lora_alpha):
    return FastLanguageModel.get_peft_model(
        model,
        r=r,
        target_modules=LORA_TARGET_MODULES,
        lora_alpha=lora_alpha,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )

def free_gpu(*objs):
    for o in objs:
        del o
    gc.collect()
    torch.cuda.empty_cache()

def run_grid_search(model_name, max_seq_length, load_in_4bit, grid,
                     search_max_steps, batch_size, grad_accum,
                     tune_train_rows, tune_dev_rows, output_dir):
    """เทรนสั้นๆ ต่อ 1 combo แล้ววัด eval_loss บน tune_dev — คืนผลเรียงจากดีสุด"""
    results = []
    for combo in grid:
        print(f"--- lr={combo['lr']}, r={combo['r']} ---")
        model, tokenizer = load_base_model(model_name, max_seq_length, load_in_4bit)
        model = add_lora(model, r=combo["r"], lora_alpha=combo["r"])

        tune_train_ds = Dataset.from_list(tune_train_rows).map(
            build_formatting_func(tokenizer), batched=True)
        tune_dev_ds = Dataset.from_list(tune_dev_rows).map(
            build_formatting_func(tokenizer), batched=True)

        trainer = SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=tune_train_ds,
            eval_dataset=tune_dev_ds,
            dataset_text_field="text",
            max_seq_length=max_seq_length,
            args=SFTConfig(
                per_device_train_batch_size=batch_size,
                per_device_eval_batch_size=batch_size,
                prediction_loss_only=True,
                gradient_accumulation_steps=grad_accum,
                max_steps=search_max_steps,
                learning_rate=combo["lr"],
                warmup_steps=5,
                fp16=not torch.cuda.is_bf16_supported(),
                bf16=torch.cuda.is_bf16_supported(),
                logging_steps=10,
                eval_strategy="steps",
                eval_steps=search_max_steps,
                save_strategy="no",
                output_dir=output_dir,
                optim="adamw_8bit",
                seed=3407,
                report_to="none",
            ),
        )
        trainer.train()
        # eval เกิดขึ้นแล้วระหว่าง train() เพราะ eval_steps == search_max_steps
        # เรียก .evaluate() ซ้ำแยกนอก train() loop จะพังใน notebook (NotebookProgressCallback bug)
        # log_history[-1] คือสรุปตอนจบเทรน (train_runtime/train_loss) ไม่มี eval_loss
        # จึงไล่หาจากท้ายมาหน้า เอาก้อนแรกที่มี key eval_loss
        eval_loss = next(
            h["eval_loss"] for h in reversed(trainer.state.log_history) if "eval_loss" in h
        )
        print(f"eval_loss = {eval_loss:.4f}")
        results.append({**combo, "eval_loss": eval_loss})

        free_gpu(model, tokenizer, trainer)

    return sorted(results, key=lambda r: r["eval_loss"])

---
## เคส A — 7-8B + QLoRA (4-bit)

โมเดล: `scb10x/typhoon2-qwen2.5-7b-instruct`

ถ้า OOM: ลด `MAX_SEQ_LENGTH_A` (ตอนนี้ 3072) เหลือ 2048, หรือลดจำนวน combo ใน `GRID_A` (ดู Stage 2.4 ในแผน)

In [ ]:
# วัดความยาว token จริงของ dataset ก่อนเทรน -> ใช้ตั้ง MAX_SEQ_LENGTH ให้พอดี
# ยาวเกิน = เปลือง VRAM + ช้า / สั้นเกิน = คำตอบโดนตัดทิ้งเงียบๆ (พังแบบไม่มี error)
from transformers import AutoTokenizer

_tok = AutoTokenizer.from_pretrained("scb10x/typhoon2-qwen2.5-7b-instruct")
_lens = sorted(
    len(_tok.apply_chat_template(to_chat_messages(r), tokenize=True))
    for r in train_rows
)
_n = len(_lens)
print(f"token length: min={_lens[0]}  p50={_lens[_n//2]}  "
      f"p90={_lens[int(_n*0.9)]}  p99={_lens[int(_n*0.99)]}  max={_lens[-1]}")
print(f"-> ตั้ง MAX_SEQ_LENGTH ประมาณ {((_lens[-1] // 256) + 1) * 256} ก็พอ (ปัดขึ้นทีละ 256)")


In [ ]:
# ==============================================================
# ⚠️  CELL นี้ OPTIONAL — ปกติ "ไม่ต้องรัน"  ⚠️
# ==============================================================
# ใช้เวลา ~25 นาที + กิน GPU quota เพื่อหา lr/r ที่ดีที่สุด
# แต่ที่ dataset 340 แถว ความต่างระหว่าง combo จมอยู่ใน noise
# -> ใช้ค่า default (BEST_A ใน cell เทรนจริง) ได้เลย ข้าม cell นี้ไป
#
# ควรกลับมารันเมื่อไหร่: ตอน dataset โตถึง ~1,000+ แถว
# ถ้าจะรัน: รันแล้วมันจะตั้ง search_results_a ให้ cell เทรนจริงหยิบไปใช้เอง
# ==============================================================
# วิธีเปิดใช้: เลือกโค้ดทั้งหมดใต้เส้นนี้ แล้วกด Ctrl+/ เพื่อ uncomment

# CASE_A_MODEL = "scb10x/typhoon2-qwen2.5-7b-instruct"
# ข้อมูลจริงยาวสุด ~3,459 ตัวอักษร (system+user+assistant) -> 3072 พอ และประหยัด VRAM กว่า 4096
# ถ้า cell วัด token บอกว่าสั้นกว่านี้ ลดลงอีกได้ = เร็วขึ้น + เสี่ยง OOM น้อยลง
# MAX_SEQ_LENGTH_A = 3072

# grid search ต้องการแค่ "อันดับ" ว่า combo ไหนดีกว่า ไม่ต้องได้โมเดลที่ดี
# -> ตัดเหลือ 3 combo โดยล็อก r=16 ไว้ (r มีผลน้อยกว่า lr มาก) แล้วกวาดเฉพาะ lr
# GRID_A = [
#     {"lr": 1e-4, "r": 16},
#     {"lr": 2e-4, "r": 16},
#     {"lr": 4e-4, "r": 16},
# ]
# SEARCH_MAX_STEPS_A = 20  # 20 step x 8 = 160 ตัวอย่าง (~0.6 epoch) พอให้ lr แยกกันชัด
# SEARCH_DEV_SIZE_A = 32   # ตอน search ใช้ dev แค่บางส่วนพอ (eval เร็วขึ้นเท่าตัว)

# SMOKE_TEST = True  -> เทสว่าโค้ดวิ่งครบ pipeline ไหม (เร็วมาก ~2-3 นาที, ผลไม่ต้องเชื่อ)
# SMOKE_TEST = False -> รันจริงเพื่อเลือก config ที่ดีสุด
# SMOKE_TEST = False

# if SMOKE_TEST:
#     grid_a = [{"lr": 2e-4, "r": 8}]   # เหลือ combo เดียว
#     search_max_steps_a = 2            # เทรนแค่ 2 step
#     train_a = tune_train_rows[:16]    # ใช้ข้อมูลนิดเดียว พอ tokenize ไว
#     dev_a = tune_dev_rows[:16]
# else:
#     grid_a = GRID_A
#     search_max_steps_a = SEARCH_MAX_STEPS_A
#     train_a = tune_train_rows
#     dev_a = tune_dev_rows[:SEARCH_DEV_SIZE_A]

# search_results_a = run_grid_search(
#     model_name=CASE_A_MODEL,
#     max_seq_length=MAX_SEQ_LENGTH_A,
#     load_in_4bit=True,
#     grid=grid_a,
#     search_max_steps=search_max_steps_a,
#     batch_size=1,
#     grad_accum=8,
#     tune_train_rows=train_a,
#     tune_dev_rows=dev_a,
#     output_dir=f"{CHECKPOINT_DIR}/case_a_search",
# )

# print("\n=== ผล grid search เคส A (เรียงจากดีสุด) ===")
# for r in search_results_a:
#     print(r)

# best_a = search_results_a[0]
# print("\nBest config (case A):", best_a)

In [ ]:
# ===== เทรนจริง — CELL นี้ต้องรัน =====
# cell นี้ยืนได้ด้วยตัวเอง ไม่ต้องรัน cell grid search ด้านบน
CASE_A_MODEL = "scb10x/typhoon2-qwen2.5-7b-instruct"
# วัดจริงจาก cell probe: max=1247 token -> 1536 เผื่อไว้ ~23% พอ
# (เดิมตั้ง 4096 = เผื่อเกินจริง 3.3 เท่า เปลือง VRAM ฟรีๆ)
MAX_SEQ_LENGTH_A = 1536

# ค่า default จาก QLoRA paper / Unsloth — ใช้เมื่อไม่ได้รัน grid search
BEST_A = {"lr": 2e-4, "r": 16}

# ถ้ารัน grid search ไว้ -> ใช้ผลจาก grid / ถ้าไม่ได้รัน -> ใช้ค่า default
best_a = search_results_a[0] if "search_results_a" in globals() else BEST_A
print("ใช้ config:", best_a)

model_a, tokenizer_a = load_base_model(CASE_A_MODEL, MAX_SEQ_LENGTH_A, load_in_4bit=True)
model_a = add_lora(model_a, r=best_a["r"], lora_alpha=best_a["r"])

# เทรนบน tune_train (272) และกัน tune_dev (68) ไว้ดูว่า overfit หรือยัง
# ยอมเสียข้อมูลเทรน 68 ใบ แลกกับการ "รู้ว่าโมเดลพังหรือเปล่า" — คุ้ม
train_ds_a = Dataset.from_list(tune_train_rows).map(
    build_formatting_func(tokenizer_a), batched=True
)
dev_ds_a = Dataset.from_list(tune_dev_rows).map(
    build_formatting_func(tokenizer_a), batched=True
)

trainer_a = SFTTrainer(
    model=model_a,
    tokenizer=tokenizer_a,
    train_dataset=train_ds_a,
    eval_dataset=dev_ds_a,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH_A,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    args=SFTConfig(
        # seq สั้นลงครึ่ง -> batch 2 ได้สบาย (2x4 = effective batch 8 เท่าเดิม แต่ GPU คุ้มขึ้น)
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=best_a["lr"],
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        # --- eval: batch เล็ก + prediction_loss_only กัน OOM (บั๊กที่เคยเจอตอน grid search) ---
        per_device_eval_batch_size=2,
        prediction_loss_only=True,
        eval_strategy="steps",
        eval_steps=20,
        # --- save ทุก 20 step ให้ตรงกับ eval แล้วโหลด checkpoint ที่ eval_loss ต่ำสุดกลับมา ---
        save_strategy="steps",
        save_steps=20,
        save_total_limit=2,          # กัน Drive เต็ม
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        output_dir=f"{CHECKPOINT_DIR}/case_a",
        optim="adamw_8bit",
        seed=3407,
        report_to="none",
    ),
)

trainer_a_stats = trainer_a.train()

# ดูว่า overfit ตอนไหน: eval_loss ควรลง แล้วถ้าเริ่มเด้งขึ้น = ท่องจำแล้ว
# (early stopping จะหยุดให้เอง และ load_best_model_at_end คืน checkpoint ที่ดีสุดมาให้)
print("\n=== eval_loss ตลอดการเทรน ===")
for h in trainer_a.state.log_history:
    if "eval_loss" in h:
        print(f"  step {h['step']:>4}: eval_loss = {h['eval_loss']:.4f}")


In [ ]:
case_a_path = f"{ADAPTER_DIR}/case_a_typhoon2_qwen25_7b"
model_a.save_pretrained(case_a_path)
tokenizer_a.save_pretrained(case_a_path)
print(f"Saved case A adapter to {case_a_path} (config: {best_a})")

**ถ้า VRAM ตึงตอนจะรันเคส B ต่อ**: Runtime → Restart runtime แล้วรันเซลล์ 0/1/2 ใหม่ ก่อนไปเซลล์เคส B ด้านล่าง (ข้ามเซลล์เคส A)

---
## เคส B — 3B + LoRA เต็ม 16-bit

โมเดล: `typhoon-ai/llama3.2-typhoon2-3b-instruct` — ไม่ quantize เพราะ 3B × 2 bytes = ~6 GB พอดี VRAM T4

**ทำไมไม่ใช้ Gemma ตามแผนเดิม** (`google/gemma-4-E4B-it`): Gemma ต้องใช้ **bf16** เพราะ activation มีค่าสูงจนทะลุเพดาน fp16 (65,504) แต่ T4 เป็นสถาปัตยกรรม Turing (2018) ซึ่ง**ไม่รองรับ bf16** (มาใน Ampere ปี 2020) Unsloth เลยถอยไปใช้ fp32 = 4 bytes/param → ~16 GB → **OOM ตั้งแต่โหลดโมเดล** แก้ด้วยการลด batch/seq len ไม่ได้เพราะน้ำหนักโมเดลอย่างเดียวก็เกินแล้ว

→ ต้องเลี่ยงโมเดลฐาน Gemma ทุกตัวบน T4 รวมถึง `typhoon2.1-gemma3-4b` ด้วย

Llama 3.2 เทรนบน NVIDIA GPU มาแต่แรก ใช้ fp16 ได้ปกติ และ Typhoon ผ่านการเทรนภาษาไทยมาแล้ว เหมาะกับงานนี้กว่า Qwen2.5-3B ดิบๆ


In [ ]:
# ==============================================================
# ⚠️  CELL นี้ OPTIONAL — ปกติ "ไม่ต้องรัน"  ⚠️
# ==============================================================
# เหตุผลเดียวกับ grid search เคส A: ที่ dataset 340 แถว ผลจมอยู่ใน noise
# -> ใช้ค่า default (BEST_B ใน cell เทรนจริง) ได้เลย
# กลับมารันเมื่อ dataset โตถึง ~1,000+ แถว
# ==============================================================
# วิธีเปิดใช้: เลือกโค้ดทั้งหมดใต้เส้นนี้ แล้วกด Ctrl+/ เพื่อ uncomment

# CASE_B_MODEL = "google/gemma-4-E4B-it"
# MAX_SEQ_LENGTH_B = 4096

# GRID_B = [
#     {"lr": 1e-4, "r": 16},
#     {"lr": 2e-4, "r": 16},
#     {"lr": 2e-4, "r": 32},
#     {"lr": 4e-4, "r": 32},
#     {"lr": 4e-4, "r": 64},
# ]
# SEARCH_MAX_STEPS_B = 40

# search_results_b = run_grid_search(
#     model_name=CASE_B_MODEL,
#     max_seq_length=MAX_SEQ_LENGTH_B,
#     load_in_4bit=False,
#     grid=GRID_B,
#     search_max_steps=SEARCH_MAX_STEPS_B,
#     batch_size=2,
#     grad_accum=4,
#     tune_train_rows=tune_train_rows,
#     tune_dev_rows=tune_dev_rows,
#     output_dir=f"{CHECKPOINT_DIR}/case_b_search",
# )

# print("\n=== ผล grid search เคส B (เรียงจากดีสุด) ===")
# for r in search_results_b:
#     print(r)

# best_b = search_results_b[0]
# print("\nBest config (case B):", best_b)

In [ ]:
# ===== เทรนจริงเคส B — CELL นี้ต้องรัน (ถ้าจะทำเคส B) =====
# เดิมแผนใช้ google/gemma-4-E4B-it แต่ Gemma ต้องการ bf16 ซึ่ง T4 ไม่มี
# -> Unsloth ถอยไป fp32 = 16 GB -> OOM ตั้งแต่โหลด (ดู markdown ด้านบน)
# Llama 3.2 ใช้ fp16 ได้ + Typhoon เทรนไทยมาแล้ว
CASE_B_MODEL = "typhoon-ai/llama3.2-typhoon2-3b-instruct"
# วัดจริงจาก cell probe (tokenizer เคส A): max=1247 -> 1536 พอ
MAX_SEQ_LENGTH_B = 1536

# ค่า default — ใช้เมื่อไม่ได้รัน grid search
BEST_B = {"lr": 2e-4, "r": 16}
best_b = search_results_b[0] if "search_results_b" in globals() else BEST_B
print("ใช้ config:", best_b)

model_b, tokenizer_b = load_base_model(CASE_B_MODEL, MAX_SEQ_LENGTH_B, load_in_4bit=False)
model_b = add_lora(model_b, r=best_b["r"], lora_alpha=best_b["r"])

# เทรนบน tune_train (272) กัน tune_dev (68) ไว้ดู overfit — เหมือนเคส A
train_ds_b = Dataset.from_list(tune_train_rows).map(
    build_formatting_func(tokenizer_b), batched=True
)
dev_ds_b = Dataset.from_list(tune_dev_rows).map(
    build_formatting_func(tokenizer_b), batched=True
)

trainer_b = SFTTrainer(
    model=model_b,
    tokenizer=tokenizer_b,
    train_dataset=train_ds_b,
    eval_dataset=dev_ds_b,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH_B,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=best_b["lr"],
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        per_device_eval_batch_size=2,
        prediction_loss_only=True,
        eval_strategy="steps",
        eval_steps=20,
        save_strategy="steps",
        save_steps=20,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        output_dir=f"{CHECKPOINT_DIR}/case_b",
        optim="adamw_8bit",
        seed=3407,
        report_to="none",
    ),
)

trainer_b_stats = trainer_b.train()

print("\n=== eval_loss ตลอดการเทรน ===")
for h in trainer_b.state.log_history:
    if "eval_loss" in h:
        print(f"  step {h['step']:>4}: eval_loss = {h['eval_loss']:.4f}")


In [ ]:
import shutil, os

# ลบโฟลเดอร์ชื่อเก่าที่ตั้งไว้ตอนแผนยังใช้ Gemma (ถ้ามี)
old_path = f"{ADAPTER_DIR}/case_b_gemma4_e4b"
if os.path.isdir(old_path):
    shutil.rmtree(old_path)
    print(f"ลบโฟลเดอร์ชื่อเก่า: {old_path}")

case_b_path = f"{ADAPTER_DIR}/case_b_typhoon2_llama32_3b"
model_b.save_pretrained(case_b_path)
tokenizer_b.save_pretrained(case_b_path)
print(f"Saved case B adapter to {case_b_path} (config: {best_b})")


---
## 3. สุ่มตรวจเร็วๆ ก่อนเข้า Stage 3 (eval เต็ม)

รันโมเดลที่เทรนเสร็จกับ 2 ตัวอย่างจาก `test.jsonl` ดูว่า JSON ออกมาสมเหตุสมผลไหม ก่อนไปทำ eval เต็มรูปแบบตาม Stage 3

In [ ]:
def quick_generate(model, tokenizer, row, max_new_tokens=400):
    FastLanguageModel.for_inference(model)
    messages = [
        {"role": "system", "content": row["system"]},
        {"role": "user", "content": row["user"]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)
    out = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens, use_cache=True)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

# เปลี่ยน model_a/tokenizer_a <-> model_b/tokenizer_b ตามเคสที่เพิ่งเทรน
CHECK_MODEL, CHECK_TOKENIZER = model_b, tokenizer_b

for sample in test_rows[:2]:
    print("=== INPUT ===")
    print(sample["user"][:200], "...")
    print("=== MODEL OUTPUT ===")
    print(quick_generate(CHECK_MODEL, CHECK_TOKENIZER, sample))
    print("=== GEMINI (ของจริงที่ใช้สอน) ===")
    print(sample["assistant"])
    print()
